# Train Gradient Boost


In [1]:
# Imports
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from code_files.train import train, train_in_batches, grid_search, random_search, save_model
from code_files.data_preperation import prepare_for_train
import pandas as pd
import numpy as np
import importlib

In [2]:
# Load Dataset
df_amazon = pd.read_csv("../../dataset/eda_amazon_sales_report.csv")
df_amazon.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117123 entries, 0 to 117122
Data columns (total 24 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Unnamed: 0                           117123 non-null  int64  
 1   Size                                 117123 non-null  int64  
 2   Qty                                  117123 non-null  int64  
 3   Amount                               117123 non-null  float64
 4   promotion-ids                        117123 non-null  int64  
 5   B2B                                  117123 non-null  int64  
 6   Status_Cancelled                     117123 non-null  bool   
 7   Status_Shipped                       117123 non-null  bool   
 8   Status_Shipped - Delivered to Buyer  117123 non-null  bool   
 9   Fulfilment_Amazon                    117123 non-null  bool   
 10  Fulfilment_Merchant                  117123 non-null  bool   
 11  ship-service-

In [3]:
# Split and Prepare for train
dftrain, dftest = train_test_split(df_amazon, test_size=0.1, random_state=42)
Xtrain_prepared, ytrain_prepared, Xtest_prepared, ytest_prepared = prepare_for_train(dftrain, dftest)

In [7]:

# Grid Search
search = random_search(
    Xtrain_prepared,
    ytrain_prepared,
    AdaBoostRegressor(),
    params={
        "estimator": [
            GradientBoostingRegressor(),
            RandomForestRegressor(),
            DecisionTreeRegressor(),
            Ridge(),
            Lasso(),
            SVR(),
        ],
        "n_estimators": [50, 100, 200, 300, 500],
        "learning_rate": [1, 1.05, 0.95],
        "loss": ["linear", "square", "exponential"]
    },
    cv=2,
    n_iters=20
)

save_model(search.best_estimator_)

Fitting 2 folds for each of 20 candidates, totalling 40 fits


c:\Users\Vincent\miniconda3\envs\gen_ml\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
3 fits failed out of a total of 40.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Vincent\miniconda3\envs\gen_ml\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\Vincent\miniconda3\envs\gen_ml\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vincent\miniconda3\envs\gen_ml\Lib\site-packages\sklearn\en

{'estimator__alpha': 1.0, 'estimator__copy_X': True, 'estimator__fit_intercept': True, 'estimator__max_iter': 1000, 'estimator__positive': False, 'estimator__precompute': False, 'estimator__random_state': None, 'estimator__selection': 'cyclic', 'estimator__tol': 0.0001, 'estimator__warm_start': False, 'estimator': Lasso(), 'learning_rate': 1.05, 'loss': 'square', 'n_estimators': 500, 'random_state': None}
Error saving model configuration to JSON file:  Object of type Lasso is not JSON serializable


In [8]:
# Show results
df_grid_results = pd.DataFrame(search.cv_results_)
columns_to_show = ["params", "rank_test_score", "mean_train_score", "mean_test_score"]
df_shown_results = df_grid_results[columns_to_show]

print(search.best_params_)
df_shown_results.sort_values("rank_test_score", ascending = True)

{'n_estimators': 500, 'loss': 'square', 'learning_rate': 1.05, 'estimator': Lasso()}


,params,rank_test_score,mean_train_score,mean_test_score
19,"{'n_estimators': 500, 'loss': 'square', 'learn...",1,-220.837187,-220.670380
2,"{'n_estimators': 100, 'loss': 'linear', 'learn...",2,-221.279315,-221.497768
7,"{'n_estimators': 200, 'loss': 'linear', 'learn...",3,-221.655274,-222.767267
15,"{'n_estimators': 50, 'loss': 'linear', 'learni...",4,-226.015985,-225.968147
14,"{'n_estimators': 100, 'loss': 'square', 'learn...",5,-244.108281,-245.666155
13,"{'n_estimators': 50, 'loss': 'exponential', 'l...",6,-245.943660,-245.711805
4,"{'n_estimators': 200, 'loss': 'square', 'learn...",7,-247.542518,-248.836048
1,"{'n_estimators': 500, 'loss': 'exponential', '...",8,-250.902592,-252.104453
6,"{'n_estimators': 50, 'loss': 'exponential', 'l...",9,-253.886032,-253.518417
18,"{'n_estimators': 300, 'loss': 'exponential', '...",10,-260.364601,-259.695061


In [9]:
# Train
model, scores = train(search.best_estimator_, Xtrain_prepared, ytrain_prepared, Xtest_prepared, ytest_prepared)
print(f"mae: {scores[0]}, rmse: {scores[1]}, r2: {scores[2]}")

mae: 346.8738415508024, rmse: 459.85485037173333, r2: 459.85485037173333


Best Param:
{'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}